# Hands-on: Episodic + procedural memory with tools

**Exercise:** Extend a LangGraph tool-calling agent so that:

1. **Episodic memory** — After tools run, write each **tool name, args, and result** (trimmed) into the LangGraph **store** under a namespace like `(user_id, "tool_episodes")`, so you can `search` later for grounding.
2. **Procedural memory** — Add a **`record_routine(name, steps)`** tool. When the model calls it, persist the routine under `(user_id, "routines")` (e.g. key `routine_<name>`). Inject saved routines into the **system prompt** on each turn.

**Graph shape:** `START → agent → (tools | END)` and when tools run: `tools → persist_memories → agent`.

**Starter code** is below. Fill in the `TODO` sections. **Solutions:** `DAY_2_HANDS_ON_langgraph_episodic_procedural_tools_SOLUTIONS.ipynb`.

**Prereq:** `NEBIUS_API_KEY` for Nebius-hosted tool-calling and embeddings (defaults: `openai/gpt-oss-120b` + `Qwen/Qwen3-Embedding-8B`).


In [1]:
%pip install -q "langgraph>=0.2" "langchain>=0.3" "langchain-openai>=0.3" "langchain-core>=0.3" "python-dotenv>=1.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 11.3 MB/s eta 0:00:00


In [2]:
%env NEBIUS_API_KEY="v1.CmQKHHN0YXRpY2tleS1lMDBlaDB4ZnhjOTJnM2FwaDESIXNlcnZpY2VhY2NvdW50LWUwMGg2Z21kNmszcWgycXRycTIMCNWdmc4GEPrCsdEBOgwI1KCxmQcQwKP10ANAAloDZTAw.AAAAAAAAAAGmLbk7vAfDFQT8zojCUKv50GQWHV5cqqdZW6MU8vPQyhpazDF___-nSCtFwJPgcvc2CPP1H6KoUO6Nb5jfcboH"

env: NEBIUS_API_KEY="v1.CmQKHHN0YXRpY2tleS1lMDBlaDB4ZnhjOTJnM2FwaDESIXNlcnZpY2VhY2NvdW50LWUwMGg2Z21kNmszcWgycXRycTIMCNWdmc4GEPrCsdEBOgwI1KCxmQcQwKP10ANAAloDZTAw.AAAAAAAAAAGmLbk7vAfDFQT8zojCUKv50GQWHV5cqqdZW6MU8vPQyhpazDF___-nSCtFwJPgcvc2CPP1H6KoUO6Nb5jfcboH"


In [3]:
from __future__ import annotations

import json
import os
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore



NEBIUS_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY", "")
if not NEBIUS_API_KEY:
    print("⚠️ NEBIUS_API_KEY not found.")
    NEBIUS_API_KEY = input("Enter your NEBIUS_API_KEY: ").strip()
NEBIUS_API_KEY = NEBIUS_API_KEY.strip().strip('"').strip("'")
assert NEBIUS_API_KEY, "Set NEBIUS_API_KEY for this exercise."

NEBIUS_CHAT_MODEL = os.getenv("NEBIUS_CHAT_MODEL", "openai/gpt-oss-120b")
NEBIUS_EMBED_MODEL = os.getenv("NEBIUS_EMBED_MODEL", "Qwen/Qwen3-Embedding-8B")

llm = ChatOpenAI(
    model=NEBIUS_CHAT_MODEL,
    api_key=NEBIUS_API_KEY,
    base_url=NEBIUS_BASE_URL,
    temperature=0,
)



def build_store() -> InMemoryStore:
    embedder = OpenAIEmbeddings(
        model=NEBIUS_EMBED_MODEL,
        api_key=NEBIUS_API_KEY,
        base_url=NEBIUS_BASE_URL,
    )
    dims = len(embedder.embed_query("dimension probe"))
    return InMemoryStore(
        index={
            "embed": embedder,
            "dims": dims,
            "fields": ["tool_name", "summary", "text", "routine_name", "steps", "$"],
        }
    )


@dataclass(frozen=True)
class AppContext:
    user_id: str


def ns_episodic(ctx: AppContext) -> tuple[str, str]:
    return (ctx.user_id, "tool_episodes")


def ns_procedural(ctx: AppContext) -> tuple[str, str]:
    return (ctx.user_id, "routines")


def _iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


@tool
def multiply_ints(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


@tool
def to_snake_case(text: str) -> str:
    """Convert a short phrase to snake_case."""
    cleaned = "".join(c.lower() if c.isalnum() or c.isspace() else " " for c in text)
    return "_".join(cleaned.split())


@tool
def record_routine(name: str, steps: str) -> str:
    """Save a named checklist for this user (procedural memory)."""
    return json.dumps({"ok": True, "routine": name, "preview": steps[:80]})


TOOLS = [multiply_ints, to_snake_case, record_routine]
tool_node = ToolNode(TOOLS)


def _last_ai_with_tool_calls(msgs: list[BaseMessage]) -> AIMessage | None:
    for m in reversed(msgs):
        if isinstance(m, AIMessage) and (m.tool_calls or []):
            return m
    return None


def _tool_results_after_ai(msgs: list[BaseMessage], ai: AIMessage) -> list[ToolMessage]:
    idx = msgs.index(ai)
    out: list[ToolMessage] = []
    for j in range(idx + 1, len(msgs)):
        if isinstance(msgs[j], ToolMessage):
            out.append(msgs[j])
        else:
            break
    return out


def persist_memories(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    """TODO: After tools run, persist:
    - For record_routine: runtime.store.put(ns_procedural(ctx), f"routine_{name}", {...})
    - For other tools: runtime.store.put(ns_episodic(ctx), new_uuid, {tool_name, args, result_preview, summary, ...})
    Match AIMessage.tool_calls to ToolMessages via tool_call_id.
    """
    raise NotImplementedError("Implement persist_memories (see solutions notebook).")


def build_system_prompt(runtime: Runtime[AppContext], user_query: str) -> str:
    """TODO: runtime.store.search episodic namespace with user_query; list procedural routines; format two blocks for the system string."""
    raise NotImplementedError("Implement build_system_prompt (see solutions notebook).")


def agent_node(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    h = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    uq = (h.content if h and isinstance(h.content, str) else "") or ""
    sys = SystemMessage(content=build_system_prompt(runtime, uq))
    bound = llm.bind_tools(TOOLS)
    reply = bound.invoke([sys, *state["messages"]])
    return {"messages": [reply]}


def route_after_agent(state: MessagesState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


# TODO: add nodes agent, tools, persist_memories; wire tools -> persist_memories -> agent
workflow = StateGraph(MessagesState, context_schema=AppContext)
# workflow.add_node(...)
# workflow.add_edge(START, "agent")
# workflow.add_conditional_edges("agent", route_after_agent, {"tools": "tools", END: END})
# workflow.add_edge("tools", "persist_memories")
# workflow.add_edge("persist_memories", "agent")

CHECKPOINTER = InMemorySaver()
STORE = build_store()
# app = workflow.compile(checkpointer=CHECKPOINTER, store=STORE)
app = None  # set after you finish the graph


BadRequestError: Error code: 400 - {'detail': 'Tokenized input is not supported'}

## Checklist

- [ ] Implement `persist_memories` (episodic for `multiply_ints` / `to_snake_case`, procedural for `record_routine`).
- [ ] Implement `build_system_prompt` using `store.search`.
- [ ] Finish `workflow` edges and `compile` with `checkpointer` + `store`.
- [ ] Run: tool math → tool string → `record_routine` → ask model to recall the routine.


In [ ]:
# Uncomment after `app` is defined:

# def run_turn(thread_id: str, user_id: str, text: str) -> dict:
#     cfg: RunnableConfig = {"configurable": {"thread_id": thread_id}}
#     return app.invoke(
#         {"messages": [HumanMessage(content=text, id=str(uuid.uuid4()))]},
#         cfg,
#         context=AppContext(user_id=user_id),
#     )

# USER, THREAD = "student-42", "session-tools-1"
# print(run_turn(THREAD, USER, "What is 21 times 7? Use multiply_ints.")["messages"][-1].content)
